In [1]:
from Bio.SeqUtils import molecular_weight as calculate_molecular_weight
import random
import itertools
import numpy as np
import pandas as pd
import gc
import multiprocessing
import os
import warnings
import time
from tqdm import tqdm
import requests
import cobra
from multiprocessing import Pool, cpu_count

from Bio.Seq import Seq
from Bio import SeqIO
from Bio.Alphabet import generic_dna

import sys
sys.path.insert(1, '../../scripts/') # comment out in python script
from utils.load_environmental_variables import *


In [2]:
# from utils import *
# from utils_2 import *
# import build_mrna_expression_reactions as build_mrna
# import build_protein_expression_reactions as build_protein
# from build_ribosome_biogenesis_reactions import ribosomal_reactions
# from build_trna_expression_reactions import trna_biogenesis_reactions

# # generate a psim from which to generate exrpession reactions

# # order the amino acids by molecular weight
# mw_map = dict()
# for k in seq_amino_acid_map_c.keys():
#     k_ = calculate_molecular_weight(k, seq_type = 'protein')
#     if k_ not in mw_map.keys():
#         mw_map[k_] = [k]
#     else:
#         mw_map[k_] += [k]
# mw_map_2 = dict()    
# for m in sorted(set(mw_map.keys())):
#     aa_code = mw_map[m]
#     if len(mw_map[m]) == 1:
#         mw_map_2[aa_code[0]] = m
#     else:
#         for aa_code_ in aa_code:
#             mw_map_2[aa_code_] = m
# mw_map = mw_map_2

# protein_lengths = [80, ptt_length, ptt_length + 1, 300,900] 
# # first three list element lengths will always be under nuclear diffusion limit
# # fourth element can be either over or under, depending on amino acids included
# # fifth will always be over nuclear diffusion limit
# # ptt_length is those that undergo post or co-translational translocation
# # we want to create two random protein sequences per situation

# max_aa_map = dict()
# for l in protein_lengths:
#     max_amino_acid = None
#     for k,v in mw_map.items():
#         if v*l < nuclear_diffusion_limit:
#             max_amino_acid = k
#         else:
#             break
#     max_aa_map[l] = max_amino_acid
# max_aa_map

# protein_lengths = [80, ptt_length, ptt_length + 1, 300,900] 

# max_aa_map = dict()
# for l in protein_lengths:
#     max_amino_acid = None
#     for k,v in mw_map.items():
#         if v*l < nuclear_diffusion_limit:
#             max_amino_acid = k
#         else:
#             break
#     max_aa_map[l] = max_amino_acid
# max_aa_map


# protein_sequences = list()
# n_sequences = 1#2
# sorted_amino_acids = list(mw_map.keys())
# for l in protein_lengths:
#     # this provides all the possible amino acids to choose to keep under or overthe nuclear diffusion limit
#     # at a given length
#     if max_aa_map[l] != None:
#         aa_under = sorted_amino_acids[:sorted_amino_acids.index(max_aa_map[l])+1]
#     else:
#         aa_under = []
#     aa_over = sorted(set(sorted_amino_acids).difference(aa_under))

#     if len(aa_over) + len(aa_under) != len(amino_acids):
#         print(len(aa_over))
#         print(len(aa_under))
#         raise ValueError('Not all amino acids are considered')
    
#     # create two protein sequences of each protein length for both those under and over nuclear diffusion limit
#     if len(aa_under) > 0:
#         for i in range(n_sequences):
#             protein_sequences.append(''.join(random.choices(aa_under, k = l)))
#     if len(aa_over) > 0:
#         for i in range(n_sequences):
#             protein_sequences.append(''.join(random.choices(aa_over, k = l)))
            
        
# if len(protein_sequences) != n_sequences*(len(protein_lengths)+1):
#     raise ValueError('Did not get all sequences')
    
    
# seq_df = pd.DataFrame(columns = ['PREMRNA_SEQ', 'MRNA_SEQ', 'PROTEIN_SEQ'])
# counter = 0
# for i in range(len(protein_sequences)):
#     ps = protein_sequences[i]
#     min_mrna_l = len(ps)*3

#     mrna_seq_min = ''.join(random.choices(['A', 'U', 'C', 'G'], k = min_mrna_l)) 
# #     mrna_seq_larger = mrna_seq_min + ''.join(random.choices(['A', 'U', 'C', 'G'], k = 200))
# #     mrna_seq_largest = mrna_seq_larger + ''.join(random.choices(['A', 'U', 'C', 'G'], k = 300))
#     mrna_seq_largest = mrna_seq_min + ''.join(random.choices(['A', 'U', 'C', 'G'], k = 500))

#     for mrna_seq in [mrna_seq_min, mrna_seq_largest]:#,mrna_seq_larger] :
#         L_mrna = len(mrna_seq)
#         pre_mrna_sequences = list()
#         premrna_seq_no_intron = mrna_seq
#         if (L_mrna + 1) * rate_intron < 1: # those with introns less than minimal length
#             premrna_seq_one_intron_A = mrna_seq + ''.join(random.choices(['A', 'U', 'C', 'G'], k = 1))
#             pre_mrna_sequences.append(premrna_seq_one_intron_A)
#         else: 
#             premrna_seq_one_intron_A = None
#         premrna_seq_one_intron_B = mrna_seq + ''.join(random.choices(['A', 'U', 'C', 'G'], k = round(1/rate_intron) - L_mrna))
# #         premrna_seq_two_introns = mrna_seq + ''.join(random.choices(['A', 'U', 'C', 'G'], k = round(2.001/rate_intron) - L_mrna))
#         premrna_seq_six_introns = mrna_seq + ''.join(random.choices(['A', 'U', 'C', 'G'], k = round(6.001/rate_intron) - L_mrna))

#         pre_mrna_sequences += [premrna_seq_one_intron_B, premrna_seq_six_introns]#, premrna_seq_two_introns]
#         for pms in pre_mrna_sequences:
#             seq_df.loc[counter, :] = [pms, mrna_seq, ps]
#             counter += 1

# if seq_df.shape[0] <  len(protein_sequences)*3*2:
#     raise ValueError('Not all combinations of sequences accounted for')

# locations = list(compartments.keys())
# locations = dict(zip(locations, ['Canonical Secretion']*len(locations)))
# for loc in locations:
#     if loc in ['n', 'c', 'x', 'm', 'i']: 
#         # mitochondrial expression not considered
#         locations[loc] = 'Cytosolic Tranport'



# psim = pd.DataFrame(columns = ['POLYA_LENGTH', 'PROTEIN_SEQ', 'MRNA_SEQ', 'PREMRNA_SEQ', 
#                   'SP', 'DSB', 'GPI', 'NG', 'OG', 'TMD', 'LOCATION', 'N_INTRONS'])
# for i in seq_df.index:
#     psim.loc[i, :] = [float('nan'), seq_df.loc[i,'PROTEIN_SEQ'], seq_df.loc[i,'MRNA_SEQ'], seq_df.loc[i,'PREMRNA_SEQ'], 
#                       True, 2, 2, float('nan'), 2, 2, list(compartments.keys()), float('nan')]
# psim['HGNC_ID'] = ['HGNC:' + str(i) for i in psim.index.tolist()]

# # get all expression reactions
# e_reactions = list()
# for hgnc_id in psim.HGNC_ID.tolist():
#     gene_info = generate_geneinfo_object(hgnc_id, psim = psim, metabolic_machinery = list(), metabolic_model = None)
#     gene_info.final_locations = locations
#     gene_info.sp = True
#     mrna_reactions, mrna_transcript_c = build_mrna.mrna_expression(gene_info)
#     protein_expression_reactions, protein_metabolites = build_protein.get_protein_expression_reactions(gene_info)
#     e_reactions += mrna_reactions + protein_expression_reactions
# e_reactions += ribosomal_reactions + trna_biogenesis_reactions + build_protein.ub_reactions


# model = cobra.Model('expression_module')
# model.add_reactions(e_reactions)
# cobra.io.json.save_json_model(model, root_path + 'expression_module_model.json')
model = cobra.io.json.load_json_model(root_path + 'expression_module_model.json')

Get minimal required metabolites for model

In [122]:
# recon2 = cobra.io.load_matlab_model(root_path + 'MammalianSecretoryRecon/MODELS/RECON2_2.mat') 
# recon2_metabolites = [m.id for m in recon2.metabolites]
# expression_module_metabolites = [m.id for m in model.metabolites]

# required_metabolites = sorted(set(expression_module_metabolites).intersection(recon2_metabolites))


# # h2o and pi
# for met_name_ in ['h2o', 'pi']:
#     for compartment in ['l', 'm', 'n', 'r', 'x']:
#         metabolite_id = met_name_ + '[' + compartment + ']'
#         if metabolite_id in required_metabolites: 
#             required_metabolites.remove(metabolite_id)

# # h
# for compartment in ['l', 'g', 'n', 'r', 'x']: # i and m must be in model for proton gradient appropriate rxns
#     metabolite_id = 'h' + '[' + compartment + ']'
#     if metabolite_id in required_metabolites: 
#         required_metabolites.remove(metabolite_id)

# #ppi
# if 'ppi[n]' in required_metabolites:
#     required_metabolites.remove('ppi[n]')

# # nucleotides
# compartments1, compartments2 = ['n'], ['l', 'm', 'r', 'x']
# for nucleotide in ['a', 'c', 'u', 'g']:
#     for phosphate in ['tp', 'dp', 'mp']:
#         if nucleotide == 'u' and phosphate == 'dp':
#             compartments = ['g', 'r', 'l']
#         elif nucleotide != 'a' or phosphate == 'mp' or nucleotide == 'g':
#             compartments = compartments1
#         else:
#             compartments = compartments1 + compartments2
#         for compartment in compartments:
#             metabolite_id = nucleotide + phosphate + '[' + compartment + ']'
#             if metabolite_id in required_metabolites: 
#                 required_metabolites.remove(metabolite_id)
    
# compartments = ['n', 'm', 'x', 'r', 'l']
# amino_acids_ = ['ala_L', 'arg_L', 'asn_L', 'asp_L', 'cys_L', 'glu_L', 'gln_L', 'gly', 'his_L', 'ile_L', 'leu_L', 
#               'lys_L', 'met_L', 'phe_L', 'pro_L', 'ser_L', 'thr_L', 'trp_L', 'tyr_L', 'val_L']
# for amino_acid in amino_acids_:
#         for compartment in compartments:
#             metabolite_id = amino_acid + '[' + compartment + ']'
#             if metabolite_id in required_metabolites: 
#                 required_metabolites.remove(metabolite_id)

# required_metabolites = sorted(set(required_metabolites + ['udp[c]', 'utp[c]', 'ctp[c]']))
# with open(root_path + 'required_metabolic_model_metabolites.txt', 'w') as f:
#     for m in required_metabolites:
#         f.write(m + '\n')

In [18]:

psim_recon2 = pd.read_csv(root_path + 'project_files/psim_recon2_2.csv', index_col = 0)
expression_machinery = sorted(open(root_path + 'project_files/expression_machinery.txt').read().splitlines())
expression_psim = psim_recon2[psim_recon2['HGNC_ID'].isin(expression_machinery)]
expression_psim.reset_index(drop = True, inplace = True)

# to do: Fix this as in build_psim_recon2_2

In [42]:
def get_premrna_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?type=genomic' 
        return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text 
    except:
        return float('nan')

def get_mrna_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?type=cdna;multiple_sequences=1' 
        return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text 
    except:
        return float('nan')

def get_protein_seq(ensg_id):
    try:
        hyperlink = 'https://rest.ensembl.org/sequence/id/' + ensg_id + '?type=protein;multiple_sequences=1' 
        return requests.get(hyperlink, headers={ "Content-Type" : "text/plain"}).text 
    except:
        return float('nan')
    
def get_all_seq(ensg_id):
    return get_premrna_seq(ensg_id), get_mrna_seq(ensg_id).splitlines()[0], get_protein_seq(ensg_id).splitlines()[0]

# formatting
def transcribe(x):
    try: 
        return str(Seq(x).transcribe())
    except:
        return float('nan')

In [57]:
missing_genes = sorted(set(expression_machinery).difference(expression_psim.HGNC_ID.tolist()))

ehm = pd.read_csv(local_data_path + 'raw/identifiers.txt', sep = '\t')
ehm = ehm.loc[ehm['NCBI gene ID'].dropna().index,:]
ehm['NCBI gene ID'] = ehm['NCBI gene ID'].astype('int64').astype(str)
ehm = ehm[ehm['HGNC ID'].isin(missing_genes)]

if len(set(missing_genes).difference(ehm['HGNC ID'].tolist() + ['ribosome'])) > 0:
    raise ValueError('Not all missing genes are maping, make sure you add them')

ens_hgnc = dict(zip(ehm['HGNC ID'].tolist(), ehm['Ensembl gene ID'].tolist()))
name_hgnc = dict(zip(ehm['HGNC ID'].tolist(), ehm['Approved symbol'].tolist()))


polyA_df = pd.read_csv(local_data_path + 'processed/polyA_length.csv', index_col = 0)

In [58]:
def get_psim_info(gene, count):
    print(count)
    ensg_id = ens_hgnc[gene]
    gene_name = name_hgnc[gene]
    
    gene_psim_info = {'GENE': gene, 'SUCCESS': 'NO'}
    
    try:
        premrna_seq, mrna_seq, protein_seq = get_all_seq(ensg_id)
        premrna_seq, mrna_seq = transcribe(premrna_seq), transcribe(mrna_seq)
        
        if type(premrna_seq) == str and type(mrna_seq) == str and type(protein_seq) == str: 
            if '{' in premrna_seq or '{' in mrna_seq or '{' in protein_seq:
                return gene_psim_info
            else:
                if gene_name in polyA_df.index:
                    polya_l = polyA_df.loc[gene_name, 'MEAN']
                else:
                    polya_l = float('nan')

                gene_psim_info['SUCCESS'] = 'YES'
                gene_psim_info['polya_l'] = polya_l
                gene_psim_info['premrna_seq'], gene_psim_info['mrna_seq'], gene_psim_info['protein_seq'] = premrna_seq, mrna_seq, protein_seq
            return gene_psim_info
        else:
            return gene_psim_info
    except:
        return gene_psim_info

In [61]:
pool = Pool(processes=(10))
missing_genes_info = pool.starmap(get_psim_info, zip(missing_genes, list(range(len(missing_genes)))))
pool.close()

In [60]:
still_missing_genes = []
counter = expression_psim.shape[0]
for i in missing_genes_info:
    if i['SUCCESS'] == 'NO':
        still_missing_genes.append(i['GENE'])
    else:
        row = [i['GENE'], i['polya_l'], i['protein_seq'], i['mrna_seq'], i['premrna_seq']]
        row += [float('nan')]*8
        
        expression_psim.loc[counter, :] = row
        counter += 1
        

/Users/joycebaghdassarian/opt/anaconda3/envs/human_me/lib/python3.6/site-packages/ipykernel_launcher.py:10 SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [69]:
# manually deal with still missing genes
gene = 'HGNC:10031'#still_missing_genes[0]
ensg_id = ens_hgnc[gene]
gene_name = name_hgnc[gene]
premrna_seq, mrna_seq, protein_seq = get_all_seq(ensg_id)
protein_seq = 'MVRYSLDPENPTKSCKSRGSNLRVHFKNTRETAQAIKGMHIRKATKYLKDVTLQKQCVPFRRYNGGVGRCAQAKQWGWTQGRWPKKSAEFLLHMLKNAESNAELKGLDVDSLVIEHIQVNKAPKMRRRTYRAHGRINPYMSSPCHIEMILTEKEQIVPKPEEEVAQKKKISQKKLKKQKLMARE'
premrna_seq, mrna_seq = transcribe(premrna_seq), transcribe(mrna_seq)



if gene_name in polyA_df.index:
    polya_l = polyA_df.loc[gene_name, 'MEAN']
else:
    polya_l = float('nan')

row = [gene, polya_l, protein_seq, mrna_seq, premrna_seq]
row += [float('nan')]*8

expression_psim.loc[expression_psim.shape[0],:] = row
###########----------
gene = 'HGNC:33511'#still_missing_genes[1]
ensg_id = 'ENST00000676383'
gene_name = name_hgnc[gene]
premrna_seq, mrna_seq, protein_seq = get_all_seq(ensg_id)
premrna_seq, mrna_seq = transcribe(premrna_seq), transcribe(mrna_seq)


if gene_name in polyA_df.index:
    polya_l = polyA_df.loc[gene_name, 'MEAN']
else:
    polya_l = float('nan')

row = [gene, polya_l, protein_seq, mrna_seq, premrna_seq]
row += [float('nan')]*8

expression_psim.loc[expression_psim.shape[0],:] = row

###########
gene = 'HGNC:31007'
ensg_id = ens_hgnc[gene]
gene_name = name_hgnc[gene]

protein_seq = 'MAAGSTTLRAVGKLQVRLATKTEPKKLEKYLQKLSALPMTADILAETGIRKTVKRLRKHQHVGDFARDLAARWKKLVLVDRNTGPDPQDPEESASRQRFGEALQEREKAWGFPENATAPRSPSHSPEHRRTARRTPPGQQRPHPRSPSREPRAERKRPRMAPADSGPHRDPPTRTAPLPMPEGPEPAVPGEQPGRGHAHAAQGGPLLGQGCQGQPQGEAVGSHSKGHKSSRGASAQKSPPVQESQSERLQAAGADSAGPKTVPSHVFSELWDPSEAWMQANYDLLSAFEAMTSQANPEALCAPALQEEAAFPGRRVNAKMPVYSGSRPACQLQVPTLRQQCLRVPRNNPDALGDVEGVPYSVLEPVLEGWTPDQLYRTEKDNAALARETDELWRIHCLQDFKEEKPQEHESWRELYLRLRDAREQRLRVVTTKIRSARENKPSGRQTKMICFNSVAKTPYDASRRQEKSAGAADPGNGEMEPAPKPAGSSQAPSGLGDGDGGSVSGGGSSNRHAAPADKTRKQAAKKVAPLMAKAIRDYKGRFSRR'
premrna_seq = 'CTTCCTACGATTCTGTCTTGCGGTGTTGTTATTTTCGCTTTTACTCCCAAGAAGATTCTCTCTGTGTCTTTTGGAATCCACAGATGTGGAAGCTGGGTACATGGAGGGCCAACGGTAGAACTCAGTGGGGTGTGTCCAGGCTGCAATTAACCTAATGACCCCGCGGGGCACAGCAGTCCACCTGGCGTGGACCGCCCCGATTGGCTGCGTTGGGCTGACGGAATGAATTCCTATCGGGGCGGGGCAGCCCAGGGGCCCAGGGGGGTCAGGGCTTGAGCCCTGGGTGGGCCCGGGGCTGGCTATATAAGGCCGGGCTCTGGCAGCGGAGCTTCACTCCGCCTTGGACGGCGCACCGCACAGGTCACACTCCAGGCTGCGGCCGCACATCCCGAGGAACAGAAGCGGGCCTGCTCAGCTGTCTGCAAGGACCGGCGTTCCAGACCAAGTGGCCCGCTGCTCCGAGGACCCAGCTCGCTCCAGGCTGTGACTCCACATCCCGAGGTCGCCGCCTGGCGACCGGGCAGCGAGGACCGGCCACCCCAGACTGCCTGTGCCGCCGCCCCGAGCTCGGACAGAGCCGCGACCGCGAGGACAGCGACGCCTGCACAGCTCTGGCGAGATGGCGGCAGGGTCCACTACGCTGCGCGCAGTGGGGAAGCTGCAGGTGCGTCTGGCCACTAAGACGGAGCCGAAAAAGCTAGAGAAATATTTGCAGAAACTCTCCGCCTTGCCCATGACCGCAGACATCCTGGCGGAGACTGGAATCAGAAAGACGGTGAAGCGCCTGCGGAAGCACCAGCACGTGGGCGACTTTGCCAGAGACTTAGCGGCCCGGTGGAAGAAGCTGGTGCTCGTGGACCGAAACACCGGGCCTGACCCGCAGGACCCTGAGGAGAGCGCTTCCCGACAGCGCTTCGGGGAGGCTCTTCAGGAGCGGGAAAAGGCCTGGGGCTTCCCAGAAAACGCGACGGCCCCCAGGAGCCCATCTCACAGCCCTGAGCACAGACGGACAGCACGCAGAACACCTCCGGGGCAACAGAGACCTCACCCGAGGTCTCCCAGTCGCGAGCCCAGAGCCGAGAGAAAGCGCCCCAGAATGGCCCCAGCTGATTCCGGCCCCCATCGGGACCCTCCAACGCGCACCGCTCCCCTCCCGATGCCCGAGGGCCCTGAGCCCGCTGTGCCCGGGGAGCAACCCGGAAGAGGCCACGCTCACGCCGCTCAGGGCGGGCCTCTGCTGGGTCAAGGCTGCCAGGGCCAACCCCAGGGGGAAGCGGTGGGGAGCCACAGCAAGGGGCACAAATCGTCCCGCGGGGCTTCGGCTCAGAAATCGCCTCCTGTCCAGGAAAGCCAGTCAGAGAGGCTGCAGGCGGCCGGCGCTGATTCCGCCGGGCCGAAAACGGTGCCCAGCCATGTCTTCTCGGAGCTCTGGGACCCCTCAGAGGCCTGGATGCAGGCCAACTACGATCTGCTGTCCGCTTTTGAGGCCATGACCTCCCAGGCAAACCCAGAAGCACTCTGCGCGCCAGCGCTCCAGGAGGAAGCTGCTTTCCCTGGACGCAGAGTGAACGCTAAGATGCCGGTGTACTCGGGCTCCAGGCCTGCCTGCCAGCTCCAGGTGCCGACGCTGCGCCAGCAGTGCCTCCGGGTGCCTAGGAACAATCCGGACGCCCTCGGCGACGTGGAAGGGGTCCCCTACTCGGTTCTTGAACCCGTTCTGGAAGGGTGGACGCCCGATCAGCTGTACCGCACAGAGAAAGACAATGCCGCACTCGCTCGAGAGACAGATGAATTATGGAGGATTCATTGCCTCCAGGACTTCAAGGAAGAAAAGCCACAGGAGCACGAGTCTTGGCGGGAGCTGTACCTGCGGCTTCGGGACGCCCGAGAGCAGCGGCTGCGAGTAGTGACCACGAAAATCCGATCCGCACGTGAAAACAAACCCAGCGGCCGACAGACAAAGATGATCTGTTTCAACTCTGTGGCCAAGACGCCTTATGATGCTTCCAGGAGGCAAGAGAAGTCTGCAGGAGCCGCTGACCCCGGAAATGGAGAGATGGAGCCAGCCCCCAAGCCCGCAGGAAGCAGCCAGGCTCCCTCCGGCCTCGGGGACGGCGACGGCGGCAGCGTGAGCGGCGGCGGCAGCAGCAACCGGCACGCGGCGCCCGCGGACAAAACCCGAAAACAGGCTGCCAAGAAAGTGGCCCCGCTGATGGCCAAGGCAATTCGAGACTACAAGGGAAGATTCTCCCGACGATAAACTCAGGACTTGCCTTACGGATAAAATCCGGGGGGAGGAGGGCCAATGCAAAGTCAATGCGGGTTGGGGAACGAAACTTGCGACGGACACCAGAACCCTTGGCTTGGTGCAAAGTTGAGCCTCCGAATCCTGCAGGTGTCAAGTGCTGGCCCTGTGATTTTTGCCTCCCACACCCAGCCAC'
premrna_seq = transcribe(premrna_seq)
mrna_seq = premrna_seq

if gene_name in polyA_df.index:
    polya_l = polyA_df.loc[gene_name, 'MEAN']
else:
    polya_l = float('nan')

row = [gene, polya_l, protein_seq, mrna_seq, premrna_seq]
row += [float('nan')]*8

expression_psim.loc[expression_psim.shape[0],:] = row


expression_psim.loc[expression_psim[expression_psim.HGNC_ID == 'HGNC:10031'].index,'PROTEIN_SEQ'] = ps = ''.join(str(Seq(expression_psim.loc[expression_psim[expression_psim.HGNC_ID == 'HGNC:10031'].index,'MRNA_SEQ'].tolist()[0]).translate()).split('*'))

In [70]:
cols = ['HGNC_ID', 'PREMRNA_SEQ', 'MRNA_SEQ', 'PROTEIN_SEQ', 'POLYA_LENGTH', 'TMD', 'SP','N_INTRONS', 'DSB', 'GPI', 'OG', 'NG',
        'LOCATION']
expression_psim['N_INTRONS'] = float('nan')
expression_psim = expression_psim[cols]

expression_psim.to_csv(root_path + 'project_files/expression_module_psim.csv')

In [2]:
expression_psim = pd.read_csv(build_files_path + 'expression_module_psim.csv', index_col = 0)
for col in ['MRNA_HALF_LIFE', 'ALPHA_P', 'PTR', 'PTR_TISSUE',  'CONSTANT_PTR']:
    expression_psim[col] = float('nan')
expression_psim.to_csv(build_files_path + 'expression_module_psim.csv')